# S6E2 CatBoost Tuning
Best default LB: **0.88447** (catboost_default).

Tuning plan:
1. **Early stopping** — find optimal iterations at lr=0.05 (lower lr, more trees)
2. **Grid search** — depth × l2_leaf_reg
3. **Refine** — rsm (random subspace) + bagging_temperature
4. **Submit** best config

## Imports & Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import catboost as cb

KAGGLE_DATA = Path("/kaggle/input/playground-series-s6e2")
LOCAL_DATA  = Path("data")
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    if "heart_disease" in df.columns:
        df["heart_disease"] = df["heart_disease"].map({"Absence": 0, "Presence": 1})
    return df

train = prep(pd.read_csv(DATA_DIR / "train.csv"))
test  = prep(pd.read_csv(DATA_DIR / "test.csv"))
ss    = pd.read_csv(DATA_DIR / "sample_submission.csv")

FEATURES     = [c for c in train.columns if c not in ["heart_disease", "id"]]
CAT_FEATURES = ["sex", "chest_pain_type", "fbs_over_120", "ekg_results",
                "exercise_angina", "slope_of_st", "number_of_vessels_fluro", "thallium"]

X = train[FEATURES]; y = train["heart_disease"]; X_test = test[FEATURES]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X.shape}  Test: {X_test.shape}")

Train: (630000, 13)  Test: (270000, 13)


## Helper: CatBoost CV with Early Stopping

In [2]:
def catboost_cv(params, X, y, cv, verbose_fold=False):
    """Manual 5-fold CV for CatBoost. Returns (mean_acc, mean_auc, best_iters)."""
    accs, aucs, best_iters = [], [], []
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = cb.CatBoostClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=0)
        best_iters.append(m.best_iteration_ or params.get('iterations', 500))
        accs.append(accuracy_score(y_val, m.predict(X_val)))
        aucs.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
        if verbose_fold:
            print(f"  fold {fold+1}: best_iter={best_iters[-1]}  acc={accs[-1]:.4f}  auc={aucs[-1]:.4f}")
    return np.mean(accs), np.mean(aucs), best_iters

## 1. Early Stopping — find optimal iterations
Lower LR (0.05) with early stopping up to 2000 trees.

In [3]:
es_params = dict(
    iterations=2000, learning_rate=0.05, depth=6,
    task_type="GPU", cat_features=CAT_FEATURES, random_state=42, verbose=0
)

es_acc, es_auc, best_iters = catboost_cv(es_params, X, y, cv, verbose_fold=True)
mean_best_iter = int(np.mean(best_iters))
print(f"\nBest iters per fold: {best_iters}")
print(f"Mean best_iter: {mean_best_iter}")
print(f"ES lr=0.05   cv_acc={es_acc:.4f}  cv_auc={es_auc:.4f}")
print(f"Default lr=0.1  cv_acc=0.8889  cv_auc=0.9554")

  fold 1: best_iter=1141  acc=0.8903  auc=0.9558
  fold 2: best_iter=1134  acc=0.8876  auc=0.9548
  fold 3: best_iter=1273  acc=0.8892  auc=0.9556
  fold 4: best_iter=1181  acc=0.8882  auc=0.9551
  fold 5: best_iter=1215  acc=0.8893  auc=0.9560

Best iters per fold: [1141, 1134, 1273, 1181, 1215]
Mean best_iter: 1188
ES lr=0.05   cv_acc=0.8889  cv_auc=0.9554
Default lr=0.1  cv_acc=0.8889  cv_auc=0.9554


## 2. Grid: depth × l2_leaf_reg
Fix iterations=mean_best_iter, lr=0.05.

In [4]:
depths    = [6, 7, 8]
l2_values = [1, 3, 5, 10]

grid_results = []
for depth in depths:
    for l2 in l2_values:
        params = dict(
            iterations=mean_best_iter, learning_rate=0.05, depth=depth,
            l2_leaf_reg=l2, task_type="GPU", cat_features=CAT_FEATURES, random_state=42, verbose=0
        )
        acc, auc, _ = catboost_cv(params, X, y, cv)
        grid_results.append({"depth": depth, "l2": l2, "cv_acc": acc, "cv_auc": auc})
        print(f"depth={depth}  l2={l2:2d}  acc={acc:.4f}  auc={auc:.4f}")

grid_df = pd.DataFrame(grid_results).sort_values("cv_acc", ascending=False)
print("\nTop 5 by cv_acc:")
print(grid_df.head().to_string(index=False))

depth=6  l2= 1  acc=0.8888  auc=0.9554
depth=6  l2= 3  acc=0.8889  auc=0.9554
depth=6  l2= 5  acc=0.8889  auc=0.9554


KeyboardInterrupt: 

## 3. Refine — rsm + bagging_temperature
Use best depth/l2. Tune random subspace method and stochastic boosting.

In [ ]:
best_row   = grid_df.iloc[0]
best_depth = int(best_row["depth"])
best_l2    = int(best_row["l2"])
print(f"Best from grid: depth={best_depth}  l2={best_l2}  cv_acc={best_row['cv_acc']:.4f}")

rsm_values     = [0.7, 0.8, 0.9, 1.0]
bagging_values = [0.5, 1.0, 1.5, 2.0]

refine_results = []
for rsm in rsm_values:
    for bt in bagging_values:
        params = dict(
            iterations=mean_best_iter, learning_rate=0.05,
            depth=best_depth, l2_leaf_reg=best_l2,
            rsm=rsm, bagging_temperature=bt,
            task_type="GPU", cat_features=CAT_FEATURES, random_state=42, verbose=0
        )
        acc, auc, _ = catboost_cv(params, X, y, cv)
        refine_results.append({"rsm": rsm, "bagging_temp": bt, "cv_acc": acc, "cv_auc": auc})
        print(f"rsm={rsm}  bt={bt}  acc={acc:.4f}  auc={auc:.4f}")

refine_df = pd.DataFrame(refine_results).sort_values("cv_acc", ascending=False)
print("\nTop 5:")
print(refine_df.head().to_string(index=False))

## 4. Best Tuned Config — Final CV & Submit

In [ ]:
best_refine = refine_df.iloc[0]
best_rsm    = best_refine["rsm"]
best_bt     = best_refine["bagging_temp"]

best_params = dict(
    iterations=mean_best_iter, learning_rate=0.05,
    depth=best_depth, l2_leaf_reg=best_l2,
    rsm=best_rsm, bagging_temperature=best_bt,
    task_type="GPU", cat_features=CAT_FEATURES, random_state=42, verbose=0
)
best_cv_acc = best_refine["cv_acc"]
best_cv_auc = best_refine["cv_auc"]

print("Best params:", {k: v for k, v in best_params.items() if k != 'cat_features'})
print(f"cv_acc={best_cv_acc:.4f}  cv_auc={best_cv_auc:.4f}")
print(f"vs default: cv_acc=0.8889  cv_auc=0.9554")

In [ ]:
def submit(model, name, cv_acc, cv_auc):
    model.fit(X, y)
    sub = ss.copy()
    sub["Heart Disease"] = model.predict_proba(X_test)[:, 1]
    fname = f"submissions/{name}.csv"
    desc  = f"{name} | cv_acc={cv_acc:.4f} | cv_auc={cv_auc:.4f}"
    sub.to_csv(fname, index=False)
    r = subprocess.run(
        ["kaggle", "competitions", "submit", "-c", "playground-series-s6e2",
         "-f", fname, "-m", desc],
        capture_output=True, text=True
    )
    status = "submitted" if "successfully" in r.stdout.lower() else r.stdout.strip()[:80]
    print(f"{name}: {status}")
    print(f"  desc: {desc}")

submit(
    cb.CatBoostClassifier(**best_params),
    "catboost_tuned_v1",
    best_cv_acc, best_cv_auc
)

## Summary

In [ ]:
summary = pd.DataFrame([
    {"model": "lr_default",        "cv_acc": 0.8831, "cv_auc": 0.9505, "lb": 0.87724},
    {"model": "xgb_default",       "cv_acc": 0.8883, "cv_auc": 0.9549, "lb": 0.88337},
    {"model": "lgbm_default",      "cv_acc": 0.8884, "cv_auc": 0.9551, "lb": 0.88421},
    {"model": "catboost_default",  "cv_acc": 0.8889, "cv_auc": 0.9554, "lb": 0.88447},
    {"model": "catboost_tuned_v1", "cv_acc": float(best_cv_acc), "cv_auc": float(best_cv_auc), "lb": None},
])
print(summary.sort_values("cv_acc", ascending=False).to_string(index=False))